# EDA 03: Promotion Lift & Discount Effectiveness Analysis

This notebook evaluates sales lift, order volume changes, and discount impact under active promotions across datasets.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

df_sales = pd.read_parquet('data/processed/sales_fact.parquet')
df_promo = pd.read_parquet('data/processed/promotion_dim.parquet')

# Drop dataset_source from promo to prevent suffix collision
df_merged = df_sales.merge(df_promo.drop(columns=['dataset_source'], errors='ignore'), on='promotion_id', how='left')
df_merged['is_promo_active'] = df_merged['promotion_id'] != 'promo_none'
print(f"Total Rows: {len(df_merged):,}")
print(df_merged['is_promo_active'].value_counts(normalize=True) * 100)


Total Rows: 1,143,942
is_promo_active
False    52.111558
True     47.888442
Name: proportion, dtype: float64


In [2]:
# Overall Promotion vs Non-Promotion Sales Metrics
promo_summary = df_merged.groupby('is_promo_active').agg(
    total_revenue=('total_sales', 'sum'),
    mean_sales=('total_sales', 'mean'),
    median_sales=('total_sales', 'median'),
    mean_quantity=('quantity', 'mean'),
    mean_discount=('discount_amount', 'mean'),
    transaction_count=('sales_id', 'count')
).reset_index()

promo_summary['status'] = np.where(promo_summary['is_promo_active'], 'Promotional', 'Baseline (No Promo)')
print(promo_summary)

no_promo_val = promo_summary.loc[~promo_summary['is_promo_active'], 'mean_sales'].values[0]
promo_val = promo_summary.loc[promo_summary['is_promo_active'], 'mean_sales'].values[0]
overall_lift = ((promo_val - no_promo_val) / no_promo_val) * 100
print(f"Overall Sales Lift from Promotions: {overall_lift:.2f}%")


   is_promo_active  total_revenue  ...  transaction_count               status
0            False   8.994190e+09  ...             596126  Baseline (No Promo)
1             True   3.617296e+09  ...             547816          Promotional

[2 rows x 8 columns]
Overall Sales Lift from Promotions: -56.24%


In [3]:
# Promotion Lift Breakdown by Dataset Source
source_promo = df_merged.groupby(['dataset_source', 'is_promo_active']).agg(
    mean_sales=('total_sales', 'mean'),
    total_revenue=('total_sales', 'sum'),
    transaction_count=('sales_id', 'count')
).reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=source_promo, x='dataset_source', y='mean_sales', hue='is_promo_active', palette='PuBuGn')
plt.title('Average Sales Revenue by Dataset Source: Promotional vs Baseline')
plt.xlabel('Dataset Source')
plt.ylabel('Mean Transaction Sales ($)')
plt.legend(title='Is Promo Active', labels=['No Promo', 'Promo'])
plt.tight_layout()
plt.show()


In [4]:
# Detailed Lift by Promotion Type
promo_type_summary = df_merged.groupby(['promotion_id', 'promo_name', 'discount_type']).agg(
    mean_sales=('total_sales', 'mean'),
    total_revenue=('total_sales', 'sum'),
    mean_discount=('discount_amount', 'mean'),
    transaction_count=('sales_id', 'count')
).reset_index()

print(promo_type_summary)

plt.figure(figsize=(10, 5))
sns.barplot(data=promo_type_summary, x='promo_name', y='mean_sales', palette='YlOrRd')
plt.title('Mean Sales Revenue per Promotion Campaign')
plt.xlabel('Promotion Name')
plt.ylabel('Mean Sales ($)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


            promotion_id  ... transaction_count
0  dataco_promo_discount  ...            170491
1       m5_promo_holiday  ...               450
2             promo_none  ...            596126
3   rossmann_promo_daily  ...            376875

[4 rows x 7 columns]
